# Baseline Authoring

This notebook walks through the eval corpus row by row so you can review and
approve AI-generated baseline commands before they are locked in.

**Workflow**
1. Run `seed-baselines` (cell below) to fill in draft commands for un-seeded rows.
2. Loop through each row, inspect the draft command, and approve or edit.
3. Each approval writes `validated_by` to the corpus row immediately.

No cell needs to run in a specific order after setup.

In [ ]:
import subprocess
import sys
from pathlib import Path

from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists()), Path.cwd())
for _p in (str(ROOT), str(ROOT / "notebooks" / "shared")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from setup import default_fixture_dir, make_skill_selector  # noqa: E402

dd_skill = make_skill_selector(ROOT)
print(f"Root: {ROOT}")
display(dd_skill)

In [ ]:
SKILL = str(dd_skill.value)
CORPUS = ROOT / f"src/skills/{SKILL}/data/eval.jsonl"
SANDBOX = ROOT / "sandbox"
FIXTURE_DIR = default_fixture_dir(SANDBOX, SKILL)

assert CORPUS.exists(), f"Corpus not found: {CORPUS}"
print("Skill: ", SKILL)
print("Corpus:", CORPUS)

## Step 1 — Generate fixtures (skip if already done)

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "knaif.evalsuite",
     "fixtures", "regen", "--skill", str(SKILL), "--sandbox", str(SANDBOX)],
    cwd=ROOT,
)
print("Return code:", result.returncode)

## Step 2 — Seed baseline commands (AI draft, not yet validated)

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "knaif.evalsuite",
     "seed-baselines", "--skill", str(SKILL), "--corpus", str(CORPUS)],
    cwd=ROOT,
    check=False,
)
print("Return code:", result.returncode)

## Step 3 — Review and approve rows

In [ ]:
from knaif.evalsuite.corpus import load_corpus, needs_review

corpus = load_corpus(CORPUS)
# Dual-mode: surfaces both un-validated single-command baselines AND un-reviewed
# multi-output (chain) rows. A chain row whose baseline was validated long ago still
# shows until its `outputs` are reviewed (it sets `outputs_validated_by` on accept).
pending = [r for r in corpus if needs_review(r)]
print(f"{len(pending)} rows awaiting validation")
for r in pending:
    kind = f"{len(r.outputs)}-step chain" if getattr(r, "outputs", None) else "single command"
    print(f"  {r.id}: {kind} | {r.utterances[0][:55]}")

In [ ]:
from baseline_reviewer import make_reviewer

REVIEWER = "human"  # set to your name or identifier

if not pending:
    print("No rows awaiting validation.")
else:
    display(make_reviewer(pending, corpus, CORPUS, FIXTURE_DIR, SANDBOX, reviewer=REVIEWER))

## Step 4 — Quick smoke test (limit=3)

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "knaif.evalsuite",
     "run", "--skill", SKILL, "--limit", "3"],
    cwd=ROOT,
)
print("Return code:", result.returncode)